In [1]:
2+2

4

In [80]:
import os
import json
from pathlib import Path

import random
import pandas as pd
import numpy as np
# import torch
import seaborn as sns
import matplotlib.pyplot as plt

import ast
import re
import shutil


In [3]:
# Setting the max characters to be displayed
pd.set_option('display.max_colwidth', None)

# display full dataframe pandas / output window widened
pd.set_option('display.width', 10000) # None

# Setting the precision for float columns
pd.set_option('display.precision', 2)

In [5]:
result_from_prev_analysis = pd.read_parquet("../results/results_myvideos_174/logits_output_meta.parquet")
print(result_from_prev_analysis.shape)
del result_from_prev_analysis['top5_classes']
result_from_prev_analysis.head(4)

(150, 12)


,video_id,cam,session,template,label,top1_pred,top1_class,top1_conf,top5_preds,correct_top1,correct_top5
0,260922-134214,cam0,plug_1,Plugging [something] into [something],49,49,Plugging [something] into [something],0.71,"[49, 106, 1, 145, 5]",True,True
1,260922-134214,cam1,plug_1,Plugging [something] into [something],49,49,Plugging [something] into [something],0.77,"[49, 106, 1, 145, 5]",True,True
2,260922-134214,cam2,plug_1,Plugging [something] into [something],49,49,Plugging [something] into [something],0.89,"[49, 1, 106, 96, 50]",True,True
3,260922-134301,cam0,plug_1,Plugging [something] into [something],49,49,Plugging [something] into [something],0.75,"[49, 106, 1, 5, 145]",True,True


In [45]:
result = pd.read_parquet("./results/predictions.parquet")
print(result.shape)
result.sample(2)

(150, 4)


,video_id,label,top1_label,correct_top1
13,260922-134433_cam1,49,49,True
148,260922-151639_cam1,101,101,True


In [46]:
# rename top1_label to top1_pred
result = result.rename(columns={'top1_label':'top1_pred'})

# I want to break video_id into video_id and cam
result[['video_id', 'cam']] = result.video_id.str.split('_', expand=True)
print(result.shape)

# Bring in session column with a left join
# Bring in template column with a left join (to later verify match rate with my custom template2id mapping.)
result_from_prev_analysis = result_from_prev_analysis[['video_id','cam', 'session','template']]
result = pd.merge(result, result_from_prev_analysis, how = 'left', on = ['video_id','cam']) # LEFT JOIN
print(result.shape)

result.sample(3)

(150, 5)
(150, 7)


,video_id,label,top1_pred,correct_top1,cam,session,template
78,260922-144425,111,15,False,cam0,books_2,Putting [something] on a surface
91,260922-150726,128,128,True,cam1,lady_1,Showing a photo of [something] to the camera
71,260922-144244,48,111,False,cam2,books_1,Piling [something] up


In [47]:
d = np.load("results/predictions.npz", allow_pickle=True)
print(d["logits"].shape)   # (150, 174)
d

(150, 174)


NpzFile 'results/predictions.npz' with keys: logits, probs, video_ids, labels

# main result

In [48]:

n = len(result)
top1_acc  = result["correct_top1"].mean()

my_metrics = {
    "total_evaluated":        n,
    "top1_accuracy":          round(top1_acc, 4),
}
my_metrics

{'total_evaluated': 150, 'top1_accuracy': np.float64(0.2067)}

- cam0 = birds eye
- cam1 = POV
- cam2 = side view

In [49]:
result.groupby("cam")[["correct_top1"]].mean().rename(
        columns={"correct_top1": "top1_acc"}
)

,top1_acc
cam,
cam0,0.20
cam1,0.34
cam2,0.08


In [50]:
result.groupby("template")[["correct_top1"]].mean().rename(
        columns={"correct_top1": "top1_acc"}
).sort_values("template")

,top1_acc
template,
Attaching [something] to [something],0.00
Moving [something] up,0.00
Piling [something] up,0.00
Plugging [something] into [something],0.80
Pushing [something] with [something],0.27
Putting [something] next to [something],0.13
Putting [something] on a surface,0.07
Putting [something] onto [something],0.07
Showing a photo of [something] to the camera,0.33


In [51]:
result[result.cam=='cam1'].groupby("template")[["correct_top1"]].mean().rename(
        columns={"correct_top1": "top1_acc"}
).sort_values("template")

,top1_acc
template,
Attaching [something] to [something],0.0
Moving [something] up,0.0
Piling [something] up,0.0
Plugging [something] into [something],1.0
Pushing [something] with [something],0.6
Putting [something] next to [something],0.2
Putting [something] on a surface,0.2
Putting [something] onto [something],0.0
Showing a photo of [something] to the camera,1.0


# Granular analysis

##### Distractor objects -> expected spike in action/verb -- P(Action|Noun) estimate
- Plug -> Plugging [something] into [something] -- 55%
- lady/person -> Showing a photo of [something] to the camera -- 96%
- magnet -> Attaching [something] to [something] -- 60%
- Books -> Piling [something] up -- 55%
- (~) Paper -> Tearing [something] into two pieces -- 25%

##### id2label

In [52]:
# Load mapping text file into a Series with template strings as the index
template2id_mapping = pd.read_csv('/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/experiment_ssv2_object_occlusion/data/template2id_mapping.txt', sep='\t', header=None, index_col=1)[0]
template2id_mapping

1
Approaching [something] with your camera                                 0
Attaching [something] to [something]                                     1
Bending [something] so that it deforms                                   2
Bending [something] until it breaks                                      3
Burying [something] in [something]                                       4
                                                                      ... 
[Something] being deflected from [something]                           169
[Something] colliding with [something] and both are being deflected    170
[Something] colliding with [something] and both come to a halt         171
[Something] falling like a feather or paper                            172
[Something] falling like a rock                                        173
Name: 0, Length: 174, dtype: int64

In [53]:
# Open text file into a dict with label string as key and integer ID as value
with open('/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/experiment_ssv2_object_occlusion/data/template2id_mapping.txt') as f:
    id2label_dict = {int(line.strip().split('\t')[0]) : line.strip().split('\t')[1] for line in f}

id2label_dict

{0: 'Approaching [something] with your camera',
 1: 'Attaching [something] to [something]',
 2: 'Bending [something] so that it deforms',
 3: 'Bending [something] until it breaks',
 4: 'Burying [something] in [something]',
 5: 'Closing [something]',
 6: 'Covering [something] with [something]',
 7: 'Digging [something] out of [something]',
 8: 'Dropping [something] behind [something]',
 9: 'Dropping [something] in front of [something]',
 10: 'Dropping [something] into [something]',
 11: 'Dropping [something] next to [something]',
 12: 'Dropping [something] onto [something]',
 13: 'Failing to put [something] into [something] because [something] does not fit',
 14: 'Folding [something]',
 15: 'Hitting [something] with [something]',
 16: 'Holding [something]',
 17: 'Holding [something] behind [something]',
 18: 'Holding [something] in front of [something]',
 19: 'Holding [something] next to [something]',
 20: 'Holding [something] over [something]',
 21: 'Laying [something] on the table on 

In [54]:
# Apply mapping to create new column
result['template_custom'] = result['label'].map(id2label_dict)
result.sample(7)

,video_id,label,top1_pred,correct_top1,cam,session,template,template_custom
59,260922-143638,101,101,True,cam2,magnet_2,Pushing [something] with [something],Pushing [something] with [something]
23,260922-141308,44,47,False,cam2,plug_2,Moving [something] up,Moving [something] up
22,260922-141308,44,18,False,cam1,plug_2,Moving [something] up,Moving [something] up
30,260922-143234,1,120,False,cam0,magnet_1,Attaching [something] to [something],Attaching [something] to [something]
93,260922-150740,128,166,False,cam0,lady_1,Showing a photo of [something] to the camera,Showing a photo of [something] to the camera
13,260922-134433,49,49,True,cam1,plug_1,Plugging [something] into [something],Plugging [something] into [something]
104,260922-150829,128,172,False,cam2,lady_1,Showing a photo of [something] to the camera,Showing a photo of [something] to the camera


In [55]:
pd.testing.assert_series_equal(result.template, result.template_custom, check_names=False, check_exact=False)

In [56]:
np.mean(result.template == result.template_custom)*100

np.float64(100.0)

In [57]:
label2id_dict = {v: k for k, v in id2label_dict.items()}

print(list(id2label_dict.items())[0:2])
print(list(label2id_dict.items())[0:2])


[(0, 'Approaching [something] with your camera'), (1, 'Attaching [something] to [something]')]
[('Approaching [something] with your camera', 0), ('Attaching [something] to [something]', 1)]


In [58]:
result.head(1)

,video_id,label,top1_pred,correct_top1,cam,session,template,template_custom
0,260922-134214,49,49,True,cam0,plug_1,Plugging [something] into [something],Plugging [something] into [something]


In [59]:
df = result[['video_id',	'cam'	,'session'	,'template',	'label'	,'top1_pred','correct_top1']]
print(df.shape)
df.head(2)

(150, 7)


,video_id,cam,session,template,label,top1_pred,correct_top1
0,260922-134214,cam0,plug_1,Plugging [something] into [something],49,49,True
1,260922-134214,cam1,plug_1,Plugging [something] into [something],49,49,True


In [60]:
# Define class probability column names
class_cols = [f'prob_{i}' for i in range(174)]

# Convert probabilities array into a DataFrame using matching index
probs_df = pd.DataFrame(d['probs'], columns=class_cols, index=df.index)

# Combine metadata with class probabilities
df = pd.concat([df, probs_df], axis=1)

pd.set_option('display.float_format',  '{:,.6f}'.format)

df.head(2)

,video_id,cam,session,template,label,top1_pred,correct_top1,prob_0,prob_1,prob_2,...,prob_164,prob_165,prob_166,prob_167,prob_168,prob_169,prob_170,prob_171,prob_172,prob_173
0,260922-134214,cam0,plug_1,Plugging [something] into [something],49,49,True,0.001100,0.042505,0.003743,...,0.003576,0.023014,0.002023,0.000389,0.004042,0.001595,0.002178,0.002823,0.000071,0.000380
1,260922-134214,cam1,plug_1,Plugging [something] into [something],49,49,True,0.000013,0.035960,0.000141,...,0.000176,0.006851,0.000031,0.000009,0.000727,0.000012,0.000013,0.000016,0.000000,0.000002


#### Main Analysis 
Comparison of:
- P(label=3|session=plug_1) -- should be high.
- P(label=3|session=plug_2) -- if its high it means plug as distractor is increasing probabilty of label=3.
- P(label=3|session not like 'plug%')

In [61]:
distractor = 'plug'
label = 3

prob_col = f'prob_{label}'

df['session_temp'] = np.where(df['session'] == f'{distractor}_1', "Normal",
                     np.where(df['session'] == f'{distractor}_2', f"distractor_{distractor}", 'Baseline'))
              
df.groupby(['session','session_temp']).agg(
    videos=('video_id', 'count'),
).reset_index()


,session,session_temp,videos
0,books_1,Baseline,15
1,books_2,Baseline,15
2,lady_1,Baseline,15
3,lady_2,Baseline,15
4,magnet_1,Baseline,15
5,magnet_2,Baseline,15
6,paper_1,Baseline,15
7,paper_2,Baseline,15
8,plug_1,Normal,15
9,plug_2,distractor_plug,15


In [62]:
pd.set_option('display.float_format',  '{:,.5f}'.format)

In [73]:
def avg_prob_in_diff_cases(df: pd.DataFrame, distractor: str, label: str, label2id_dict: dict, id2label_dict:dict, cams: list):
    label_id = label2id_dict[label]
    prob_col = f'prob_{label_id}'

    df_temp = df[df['cam'].isin(cams)].copy()
    df_temp['session_temp'] = np.where(df_temp['session'] == f'{distractor}_1', "Normal",
                        np.where(df_temp['session'] == f'{distractor}_2', f"distractor_{distractor}", 'Baseline'))
    
    r = df_temp.groupby('session_temp').agg(
        videos=('video_id', 'count'),
        prob_mean=(prob_col, 'mean'),
        prob_std=(prob_col, 'std'),
        top1acc=('correct_top1', 'mean'),
        examples=('video_id', lambda x: ', '.join(map(str, x.unique()[:3]))),
    ).sort_values('session_temp').reset_index()

    print("=" * 25, distractor, "=" * 25)
    print(f"label {label_id}: {label}")
    print(f"cams used: {cams}")
    print(r)
    
    print("=" * 20, 'Videos with distractor', "=" * 20)
    dist_df = df_temp[df_temp['session_temp'] == f"distractor_{distractor}"]
    prob_cols = [c for c in dist_df.columns if c.startswith('prob_')]
    meta_cols = [c for c in dist_df.columns if c not in prob_cols]

    for idx, row in dist_df.iterrows():
        top5_s = row[prob_cols].astype(float).nlargest(5)
        top5_cols = top5_s.index.tolist()
        
        # 1. Print row table as before (meta columns + top 5 prob_X headers)
        print(row[meta_cols + top5_cols].to_frame().T)
        
        # 2. Print mapping and probability cleanly in 5 separate lines
        print("\nTop 5 Probability Breakdown:")
        for col, val in top5_s.items():
            class_id = int(col.split('_')[1])
            class_name = id2label_dict.get(class_id, id2label_dict.get(str(class_id), 'Unknown'))
            print(f"  {col:<8} ->  {val:.5f}  |  {class_name}")

        print(f"\n  {prob_col:<8} ->  {row[prob_col]:.5f}  |  {label}")
            
        print("-" * 50)

In [74]:
avg_prob_in_diff_cases(df, distractor='plug', label='Plugging [something] into [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= plug =========================
label 49: Plugging [something] into [something]
cams used: ['cam1']
      session_temp  videos  prob_mean  prob_std  top1acc                                     examples
0         Baseline      40    0.00166   0.00676  0.30000  260922-143234, 260922-143255, 260922-143323
1           Normal       5    0.54720   0.16017  1.00000  260922-134214, 260922-134301, 260922-134333
2  distractor_plug       5    0.00123   0.00146  0.00000  260922-141236, 260922-141249, 260922-141308
==================== Videos with distractor ====================
         video_id   cam session                                 template label top1_pred correct_top1     session_temp prob_73 prob_109 prob_120 prob_19 prob_101
16  260922-141236  cam1  plug_2  Putting [something] next to [something]   109        73        False  distractor_plug 0.23819  0.19018  0.16505 0.08087  0.03878

Top 5 Probability Breakdown:
  prob_73  ->  0.23819  |  Pretending to put [so

In [75]:
avg_prob_in_diff_cases(df, distractor='lady', label='Showing a photo of [something] to the camera', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= lady =========================
label 128: Showing a photo of [something] to the camera
cams used: ['cam1']
      session_temp  videos  prob_mean  prob_std  top1acc                                     examples
0         Baseline      40    0.00141   0.00532  0.30000  260922-134214, 260922-134301, 260922-134333
1           Normal       5    0.74318   0.21443  1.00000  260922-150726, 260922-150740, 260922-150754
2  distractor_lady       5    0.00034   0.00046  0.00000  260922-150856, 260922-150915, 260922-150927
==================== Videos with distractor ====================
          video_id   cam session                                 template label top1_pred correct_top1     session_temp prob_124 prob_11 prob_125 prob_109 prob_120
106  260922-150856  cam1  lady_2  Putting [something] next to [something]   109       124        False  distractor_lady  0.27580 0.13647  0.10904  0.08557  0.06019

Top 5 Probability Breakdown:
  prob_124 ->  0.27580  |  Showing [

In [76]:
avg_prob_in_diff_cases(df, distractor='magnet', label='Attaching [something] to [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= magnet =========================
label 1: Attaching [something] to [something]
cams used: ['cam1']
        session_temp  videos  prob_mean  prob_std  top1acc                                     examples
0           Baseline      40    0.00961   0.02416  0.35000  260922-134214, 260922-134301, 260922-134333
1             Normal       5    0.01627   0.02287  0.00000  260922-143234, 260922-143255, 260922-143323
2  distractor_magnet       5    0.00115   0.00065  0.60000  260922-143515, 260922-143535, 260922-143557
==================== Videos with distractor ====================
         video_id   cam   session                                 template label top1_pred correct_top1       session_temp prob_109 prob_73 prob_101 prob_41 prob_36
46  260922-143515  cam1  magnet_2  Putting [something] next to [something]   109       109         True  distractor_magnet  0.24566 0.21136  0.15977 0.05286 0.03612

Top 5 Probability Breakdown:
  prob_109 ->  0.24566  |  Putting

In [77]:
avg_prob_in_diff_cases(df, distractor='books', label='Piling [something] up', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= books =========================
label 48: Piling [something] up
cams used: ['cam1']
       session_temp  videos  prob_mean  prob_std  top1acc                                     examples
0          Baseline      40    0.00056   0.00109  0.40000  260922-134214, 260922-134301, 260922-134333
1            Normal       5    0.02625   0.01797  0.00000  260922-144210, 260922-144221, 260922-144233
2  distractor_books       5    0.00068   0.00079  0.20000  260922-144349, 260922-144425, 260922-144448
==================== Videos with distractor ====================
         video_id   cam  session                                 template label top1_pred correct_top1      session_temp prob_73 prob_109 prob_120 prob_101 prob_19
76  260922-144349  cam1  books_2  Putting [something] next to [something]   109        73        False  distractor_books 0.45761  0.33265  0.04776  0.03102 0.02037

Top 5 Probability Breakdown:
  prob_73  ->  0.45761  |  Pretending to put [something

##### Looking at same close analysis but for Normal videos now. lets see books

In [81]:
def avg_prob_in_diff_cases(df: pd.DataFrame, distractor: str, label: str, label2id_dict: dict, id2label_dict:dict, cams: list):
    label_id = label2id_dict[label]
    prob_col = f'prob_{label_id}'

    df_temp = df[df['cam'].isin(cams)].copy()
    df_temp['session_temp'] = np.where(df_temp['session'] == f'{distractor}_1', "Normal",
                        np.where(df_temp['session'] == f'{distractor}_2', f"distractor_{distractor}", 'Baseline'))
    
    r = df_temp.groupby('session_temp').agg(
        videos=('video_id', 'count'),
        prob_mean=(prob_col, 'mean'),
        prob_std=(prob_col, 'std'),
        top1acc=('correct_top1', 'mean'),
        examples=('video_id', lambda x: ', '.join(map(str, x.unique()[:3]))),
    ).sort_values('session_temp').reset_index()

    print("=" * 25, distractor, "=" * 25)
    print(f"label {label_id}: {label}")
    print(f"cams used: {cams}")
    print(r)
    print("=" * 20, 'Videos with distractor', "=" * 20)

    dist_df = df_temp[df_temp['session_temp'] == f"Normal"]

    # ------------- chunk to copy videos ------------- #
    source_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/vjepa_evaluation_on_ssv2/apply_dinov2_attn_probe/data/frames/"
    dest_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/vjepa_evaluation_on_ssv2/analysis_videos/"

    # 2. Clear destination directory if it exists, then recreate it
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)
    os.makedirs(dest_dir, exist_ok=True)

    # 3. Get unique video IDs and copy matching files
    unique_video_ids = dist_df['video_id'].dropna().astype(str).unique()

    for filename in os.listdir(source_dir):
        if any(video_id in filename for video_id in unique_video_ids):
            shutil.copy(os.path.join(source_dir, filename), os.path.join(dest_dir, filename))
    # ------------------------------------------------ #

    prob_cols = [c for c in dist_df.columns if c.startswith('prob_')]
    meta_cols = [c for c in dist_df.columns if c not in prob_cols]

    for idx, row in dist_df.iterrows():
        top5_s = row[prob_cols].astype(float).nlargest(5)
        top5_cols = top5_s.index.tolist()
        
        # 1. Print row table as before (meta columns + top 5 prob_X headers)
        print(row[meta_cols + top5_cols].to_frame().T)
        
        # 2. Print mapping and probability cleanly in 5 separate lines
        print("\nTop 5 Probability Breakdown:")
        for col, val in top5_s.items():
            class_id = int(col.split('_')[1])
            class_name = id2label_dict.get(class_id, id2label_dict.get(str(class_id), 'Unknown'))
            print(f"  {col:<8} ->  {val:.5f}  |  {class_name}")

        print(f"\n  {prob_col:<8} ->  {row[prob_col]:.5f}  |  {label}")
            
        print("-" * 50)

In [82]:
avg_prob_in_diff_cases(df, distractor='books', label='Piling [something] up', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= books =========================
label 48: Piling [something] up
cams used: ['cam1']
       session_temp  videos  prob_mean  prob_std  top1acc                                     examples
0          Baseline      40    0.00056   0.00109  0.40000  260922-134214, 260922-134301, 260922-134333
1            Normal       5    0.02625   0.01797  0.00000  260922-144210, 260922-144221, 260922-144233
2  distractor_books       5    0.00068   0.00079  0.20000  260922-144349, 260922-144425, 260922-144448
==================== Videos with distractor ====================
         video_id   cam  session               template label top1_pred correct_top1 session_temp prob_124 prob_161  prob_0 prob_126 prob_162
61  260922-144210  cam1  books_1  Piling [something] up    48       124        False       Normal  0.26491  0.06937 0.05851  0.04551  0.03795

Top 5 Probability Breakdown:
  prob_124 ->  0.26491  |  Showing [something] behind [something]
  prob_161 ->  0.06937  |  Turnin

In [83]:
avg_prob_in_diff_cases(df, distractor='magnet', label='Attaching [something] to [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= magnet =========================
label 1: Attaching [something] to [something]
cams used: ['cam1']
        session_temp  videos  prob_mean  prob_std  top1acc                                     examples
0           Baseline      40    0.00961   0.02416  0.35000  260922-134214, 260922-134301, 260922-134333
1             Normal       5    0.01627   0.02287  0.00000  260922-143234, 260922-143255, 260922-143323
2  distractor_magnet       5    0.00115   0.00065  0.60000  260922-143515, 260922-143535, 260922-143557
==================== Videos with distractor ====================
         video_id   cam   session                              template label top1_pred correct_top1 session_temp prob_104 prob_19 prob_109 prob_120 prob_15
31  260922-143234  cam1  magnet_1  Attaching [something] to [something]     1       104        False       Normal  0.14434 0.09036  0.08452  0.06283 0.05376

Top 5 Probability Breakdown:
  prob_104 ->  0.14434  |  Putting [something that

In [84]:
avg_prob_in_diff_cases(df, distractor='paper', label='Tearing [something] into two pieces', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= paper =========================
label 144: Tearing [something] into two pieces
cams used: ['cam1']
       session_temp  videos  prob_mean  prob_std  top1acc                                     examples
0          Baseline      40    0.00200   0.00646  0.35000  260922-134214, 260922-134301, 260922-134333
1            Normal       5    0.42439   0.22271  0.40000  260922-151251, 260922-151331, 260922-151352
2  distractor_paper       5    0.00056   0.00086  0.20000  260922-151521, 260922-151539, 260922-151607
==================== Videos with distractor ====================
          video_id   cam  session                             template label top1_pred correct_top1 session_temp prob_145 prob_144 prob_65  prob_2 prob_165
121  260922-151251  cam1  paper_1  Tearing [something] into two pieces   144       145        False       Normal  0.55522  0.39143 0.02686 0.00718  0.00546

Top 5 Probability Breakdown:
  prob_145 ->  0.55522  |  Tearing [something] just a li

### A closer look
##### at individual examples and try to spot the ones where the distractor seems to work.
##### ***Beware that sigmoid_logits are used as probability (PROBLEM IS THAT TOP5 ALWAYS HAVE 99% PROB)

In [126]:
df = result[['video_id',	'cam'	,'session'	,'template',	'label'	,'top1_pred','top1_conf','correct_top1']]
print(df.shape)
df.head(2)

(150, 8)


,video_id,cam,session,template,label,top1_pred,top1_conf,correct_top1
0,260922-134214,cam0,plug_1,Plugging [something] into [something],49,49,0.70513,True
1,260922-134214,cam1,plug_1,Plugging [something] into [something],49,49,0.77079,True


In [127]:
def sigmoid(x):
    return 1/(1+np.exp(-x))
    
d_sigmoid_logits = sigmoid(d['logits'])
d_sigmoid_logits.shape

(150, 174)

In [128]:
# Define class probability column names
class_cols = [f'prob_{i}' for i in range(174)]

# Convert probabilities array into a DataFrame using matching index
# probs_df = pd.DataFrame(d['probs'], columns=class_cols, index=df.index)
probs_df = pd.DataFrame(d_sigmoid_logits, columns=class_cols, index=df.index)

# Combine metadata with class probabilities
df = pd.concat([df, probs_df], axis=1)

pd.set_option('display.float_format',  '{:,.6f}'.format)

df.head(2)

,video_id,cam,session,template,label,top1_pred,top1_conf,correct_top1,prob_0,prob_1,...,prob_164,prob_165,prob_166,prob_167,prob_168,prob_169,prob_170,prob_171,prob_172,prob_173
0,260922-134214,cam0,plug_1,Plugging [something] into [something],49,49,0.705132,True,0.106691,0.999704,...,0.460774,0.125065,0.370225,0.171061,0.214691,0.597843,0.924142,0.004070,0.000536,0.103749
1,260922-134214,cam1,plug_1,Plugging [something] into [something],49,49,0.770786,True,0.205615,0.999722,...,0.496765,0.189521,0.320821,0.169956,0.240775,0.337204,0.933431,0.005555,0.000688,0.174410


In [123]:
def avg_prob_in_diff_cases(df: pd.DataFrame, distractor: str, label: str, label2id_dict: dict, id2label_dict:dict, cams: list):
    label_id = label2id_dict[label]
    prob_col = f'prob_{label_id}'

    df_temp = df[df['cam'].isin(cams)].copy()
    df_temp['session_temp'] = np.where(df_temp['session'] == f'{distractor}_1', "Normal",
                        np.where(df_temp['session'] == f'{distractor}_2', f"distractor_{distractor}", 'Baseline'))
    
    r = df_temp.groupby('session_temp').agg(
        videos=('video_id', 'count'),
        prob_mean=(prob_col, 'mean'),
        prob_std=(prob_col, 'std'),
        top1acc=('correct_top1', 'mean'),
        examples=('video_id', lambda x: ', '.join(map(str, x.unique()[:3]))),
    ).sort_values('session_temp').reset_index()

    print("=" * 25, distractor, "=" * 25)
    print(f"label {label_id}: {label}")
    print(f"cams used: {cams}")
    print(r)
    
    print("=" * 20, 'Videos with distractor', "=" * 20)
    dist_df = df_temp[df_temp['session_temp'] == f"distractor_{distractor}"]
    prob_cols = [c for c in dist_df.columns if c.startswith('prob_')]
    meta_cols = [c for c in dist_df.columns if c not in prob_cols]

    for idx, row in dist_df.iterrows():
        top5_cols = row[prob_cols].astype(float).nlargest(5).index.tolist()
        print(row[meta_cols + top5_cols].to_frame().T)
        print('-'*10)

avg_prob_in_diff_cases(df, distractor='plug', label='Plugging [something] into [something]', label2id_dict=label2id_dict, cams=['cam1'])

========================= plug =========================
label 49: Plugging [something] into [something]
cams used: ['cam1']
      session_temp  videos  prob_mean  prob_std  top1acc                                     examples
0         Baseline      40    0.64286   0.22287  0.55000  260922-143234, 260922-143255, 260922-143323
1           Normal       5    0.99996   0.00002  1.00000  260922-134214, 260922-134301, 260922-134333
2  distractor_plug       5    0.40757   0.27606  0.80000  260922-141236, 260922-141249, 260922-141308
==================== Videos with distractor ====================
         video_id   cam session                                 template label top1_pred top1_conf correct_top1     session_temp prob_114 prob_107 prob_42  prob_6 prob_105
16  260922-141236  cam1  plug_2  Putting [something] next to [something]   107       114   0.70823        False  distractor_plug  0.99975  0.99887 0.99265 0.98999  0.98795
----------
         video_id   cam session                

In [124]:
def avg_prob_in_diff_cases(df: pd.DataFrame, distractor: str, label: str, label2id_dict: dict, id2label_dict:dict, cams: list):
    label_id = label2id_dict[label]
    prob_col = f'prob_{label_id}'

    df_temp = df[df['cam'].isin(cams)].copy()
    df_temp['session_temp'] = np.where(df_temp['session'] == f'{distractor}_1', "Normal",
                        np.where(df_temp['session'] == f'{distractor}_2', f"distractor_{distractor}", 'Baseline'))
    
    r = df_temp.groupby('session_temp').agg(
        videos=('video_id', 'count'),
        prob_mean=(prob_col, 'mean'),
        prob_std=(prob_col, 'std'),
        top1acc=('correct_top1', 'mean'),
        examples=('video_id', lambda x: ', '.join(map(str, x.unique()[:3]))),
    ).sort_values('session_temp').reset_index()

    print("-" * 25, distractor, "-" * 25)
    print(f"label {label_id}: {label}")
    print(f"cams used: {cams}")
    print(r)
    
    print("-" * 20, 'Videos with distractor', "-" * 20)
    dist_df = df_temp[df_temp['session_temp'] == f"distractor_{distractor}"]
    prob_cols = [c for c in dist_df.columns if c.startswith('prob_')]
    meta_cols = [c for c in dist_df.columns if c not in prob_cols]

    for idx, row in dist_df.iterrows():
        top5_cols = row[prob_cols].astype(float).nlargest(5).index.tolist()
        
        # Rename top 5 prob columns to include full class label name
        rename_map = {}
        for col in top5_cols:
            class_id = int(col.split('_')[1])
            class_name = id2label_dict.get(class_id, id2label_dict.get(str(class_id), col))
            rename_map[col] = f"{col}: {class_name}"

        row_df = row[meta_cols + top5_cols].to_frame().T.rename(columns=rename_map)
        print(row_df)
        print("-" * 50)

avg_prob_in_diff_cases(df, distractor='plug', label='Plugging [something] into [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

------------------------- plug -------------------------
label 49: Plugging [something] into [something]
cams used: ['cam1']
      session_temp  videos  prob_mean  prob_std  top1acc                                     examples
0         Baseline      40    0.64286   0.22287  0.55000  260922-143234, 260922-143255, 260922-143323
1           Normal       5    0.99996   0.00002  1.00000  260922-134214, 260922-134301, 260922-134333
2  distractor_plug       5    0.40757   0.27606  0.80000  260922-141236, 260922-141249, 260922-141308
-------------------- Videos with distractor --------------------
         video_id   cam session                                 template label top1_pred top1_conf correct_top1     session_temp prob_114: Putting [something similar to other things that are already on the table] prob_107: Putting [something] next to [something] prob_42: Moving [something] closer to [something] prob_6: Covering [something] with [something] prob_105: Putting [something] in front of [